# 19｜大型語言模型_生成與聊天系統

在固定 logits 上比較 greedy 與 top-k 抽樣，觀察溫度的影響。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

rng = np.random.default_rng(SEED)


## 課堂小實驗

In [ ]:
vocab = np.array(['資料', '模型', '學習', '風險', '答案'])
logits = np.array([1.2, 2.4, 1.8, 0.4, 1.0])
def probs(temp):
    z = logits / temp; p = np.exp(z-z.max()); return p/p.sum()
def sample_top_k(temp=1.0, k=3, n=8):
    p = probs(temp); ids = np.argsort(p)[-k:]; q = p[ids]/p[ids].sum()
    return rng.choice(vocab[ids], size=n, p=q)
rows = pd.DataFrame({'token': vocab, 'p_T0.7': probs(.7), 'p_T1.3': probs(1.3)})
print('greedy:', vocab[np.argmax(logits)])
print('top-k sample:', ' '.join(sample_top_k()))
rows.round(3)

## 執行結果圖

In [ ]:
x = np.arange(len(vocab)); w = .36
plt.bar(x-w/2, probs(.7), w, label='temperature 0.7')
plt.bar(x+w/2, probs(1.3), w, label='temperature 1.3')
plt.xticks(x, vocab); plt.ylabel('probability'); plt.title('Temperature changes distribution sharpness')
plt.legend()
savefig('19_sampling_demo')
report('llm_sampling', {'vocab': vocab.tolist(), 'p_07': probs(.7).tolist(), 'p_13': probs(1.3).tolist()})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。